# HDFS Experiments v2 Report Dashboard

Notebook này đọc report CSV mới nhất trong `experiments_v2/results_protocol_v4/` và trực quan hóa kết quả theo nhiều góc nhìn: tổng quan, fixed top-k sweep, adaptive vs fixed, KB variants, và confusion matrix.

In [ ]:
from pathlib import Path
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

RESULTS_DIR = Path('experiments_v2/results_protocol_v4')
SPLIT = 'validation'


def version_number(path):
    match = re.search(r'_v(\d+)\.csv$', path.name)
    return int(match.group(1)) if match else 1

candidates = sorted(RESULTS_DIR.glob(f'comparison_{SPLIT}*.csv'), key=version_number)
if not candidates:
    raise FileNotFoundError(f'No comparison_{SPLIT}*.csv found in {RESULTS_DIR}')
REPORT_PATH = candidates[-1]
df = pd.read_csv(REPORT_PATH)
print('Loaded:', REPORT_PATH)
print('Rows:', len(df))
display(df.head())

## 1. Bảng đầy đủ

Đây là report đầy đủ đã sort sẵn theo method/KB/context.

In [ ]:
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 120)
display(df)

## 2. Best overall và best trong từng method/KB

In [ ]:
best_overall = df[df['best_overall'] == True].copy()
print('Best overall')
display(best_overall[['model_profile','method','kb','context_setting','f1','precision','recall','confusion_matrix']])

best_group = df[df['best_in_method_kb'] == True].copy()
cols = ['method','retrieval','selection','kb','context_setting','contexts','f1','precision','recall','average_contexts','confusion_matrix']
display(best_group[cols].sort_values(['method','kb']).reset_index(drop=True))

## 3. Fixed top-k sweep: F1 theo số context

Biểu đồ này cho biết mỗi retrieval/KB đạt F1 tốt nhất ở top-k nào.

In [ ]:
fixed = df[df['selection'] == 'fixed'].copy()
fixed = fixed.dropna(subset=['contexts'])
fixed['contexts'] = fixed['contexts'].astype(int)

for method in sorted(fixed['method'].dropna().unique()):
    subset = fixed[fixed['method'] == method]
    if subset.empty:
        continue
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for kb, group in subset.groupby('kb', dropna=False):
        group = group.sort_values('contexts')
        ax.plot(group['contexts'], group['f1'], marker='o', label=str(kb))
    ax.set_title(f'{method.upper()} fixed top-k sweep')
    ax.set_xlabel('fixed top-k contexts')
    ax.set_ylabel('F1')
    ax.set_xticks(sorted(subset['contexts'].unique()))
    ax.grid(True, alpha=0.3)
    ax.legend(title='KB')
    plt.show()

## 4. Adaptive so với best fixed cùng retrieval/KB

In [ ]:
adaptive = df[df['selection'] == 'adaptive'].copy()
cols = ['method','retrieval','kb','context_setting','average_contexts','f1','best_fixed_context_setting','best_fixed_f1','delta_f1_vs_best_fixed','confusion_matrix']
display(adaptive[cols].sort_values(['method','kb']).reset_index(drop=True))

plot_df = adaptive.dropna(subset=['best_fixed_f1']).copy()
if not plot_df.empty:
    labels = plot_df.apply(lambda r: f"{r['method']} {r['kb']}", axis=1)
    x = np.arange(len(plot_df))
    width = 0.38
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(x - width/2, plot_df['best_fixed_f1'], width, label='best fixed')
    ax.bar(x + width/2, plot_df['f1'], width, label='adaptive')
    ax.set_xticks(x, labels, rotation=35, ha='right')
    ax.set_ylabel('F1')
    ax.set_title('Adaptive vs best fixed baseline')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 5. Heatmap F1 theo method và KB

Với fixed, lấy dòng tốt nhất trong mỗi method/KB. Adaptive chỉ có một dòng mỗi method/KB.

In [ ]:
heat = df[df['best_in_method_kb'] == True].copy()
heat['label'] = heat.apply(lambda r: f"{r['method']}:{r['context_setting']}", axis=1)
pivot = heat.pivot_table(index='label', columns='kb', values='f1', aggfunc='max')
display(pivot)

fig, ax = plt.subplots(figsize=(8, max(3, 0.45 * len(pivot))))
values = pivot.values.astype(float)
im = ax.imshow(values, cmap='YlGnBu', aspect='auto')
ax.set_xticks(np.arange(len(pivot.columns)), pivot.columns)
ax.set_yticks(np.arange(len(pivot.index)), pivot.index)
for i in range(values.shape[0]):
    for j in range(values.shape[1]):
        if not np.isnan(values[i, j]):
            ax.text(j, i, f'{values[i, j]:.3f}', ha='center', va='center', fontsize=9)
ax.set_title('Best F1 by method/context and KB')
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## 6. Confusion matrix viewer

Chọn một dòng theo `method`, `kb`, `context_setting` rồi vẽ confusion matrix.

In [ ]:
def show_confusion(method, kb=None, context_setting=None):
    subset = df[df['method'] == method].copy()
    if kb is not None:
        subset = subset[subset['kb'].fillna('') == kb]
    if context_setting is not None:
        subset = subset[subset['context_setting'].fillna('') == context_setting]
    if subset.empty:
        print('No matching row')
        return
    row = subset.iloc[0]
    matrix = np.array([[row['TN'], row['FP']], [row['FN'], row['TP']]], dtype=int)
    fig, ax = plt.subplots(figsize=(4.5, 3.8))
    im = ax.imshow(matrix, cmap='Blues')
    ax.set_xticks([0,1], ['Pred NORMAL', 'Pred ANOMALY'])
    ax.set_yticks([0,1], ['True NORMAL', 'True ANOMALY'])
    title_bits = [str(row['method'])]
    if pd.notna(row.get('kb')): title_bits.append(str(row['kb']))
    if pd.notna(row.get('context_setting')): title_bits.append(str(row['context_setting']))
    ax.set_title(' / '.join(title_bits) + f" | F1={row['f1']:.3f}")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(matrix[i, j]), ha='center', va='center')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()
    display(row.to_frame('value'))

# Ví dụ:
show_confusion('m5', 'normal', 'adaptive')

## 7. Context reduction và latency/token cost

In [ ]:
cost_cols = ['method','kb','context_setting','f1','average_contexts','context_reduction_vs_fixed_contexts','average_prompt_tokens','average_latency_seconds_uncached']
display(df[cost_cols].sort_values(['method','kb','average_contexts']).reset_index(drop=True))

fig, ax = plt.subplots(figsize=(7, 4.5))
for selection, group in df.groupby('selection', dropna=False):
    ax.scatter(group['average_contexts'], group['f1'], label=str(selection), alpha=0.8)
ax.set_xlabel('Average contexts')
ax.set_ylabel('F1')
ax.set_title('F1 vs average context count')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()